# Chapter 2: Your First DataFrame

Chapter 1 explained what pandas is and where it fits. Now you write real code. You will create a DataFrame from scratch, inspect it, spot common data-type traps, export a result, and learn what to do when things break.

In [1]:
import pandas as pd
print(f'pandas version: {pd.__version__}')

pandas version: 2.3.3


---

## 2.1 Creating Your First DataFrame

You have just started as a data analyst at Acme Corp. Your manager hands you the employee data and says: "Get familiar with this."

In [2]:
employees = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana ', 'Eve'],
    'department': ['Sales', 'Engineering', 'Sales', 'HR', 'Engineering'],
    'salary': [50000, 75000, 55000, 62000, None],
    'hire_date': ['2021-03-15', '2019-07-01', '2022-01-10', '2020-11-20', '2023-06-05'],
    'city': ['Denver', 'Austin', 'Denver', 'Chicago', 'Austin']
})

print(employees)

      name   department   salary   hire_date     city
0    Alice        Sales  50000.0  2021-03-15   Denver
1      Bob  Engineering  75000.0  2019-07-01   Austin
2  Charlie        Sales  55000.0  2022-01-10   Denver
3   Diana            HR  62000.0  2020-11-20  Chicago
4      Eve  Engineering      NaN  2023-06-05   Austin


**What to notice:** Two problems are hiding in plain sight. Eve's salary is `NaN` (she was entered with `None`), and Diana's name has a trailing space — `'Diana '` instead of `'Diana'`. Real data has these issues. We will not fix them yet — Chapter 10 covers missing values and Chapter 16 covers string cleaning — but noticing them now is the skill.

In [3]:
# Three structural attributes to check immediately
print('Shape:', employees.shape)
print('Columns:', employees.columns.tolist())
print('Index:', employees.index)

Shape: (5, 5)
Columns: ['name', 'department', 'salary', 'hire_date', 'city']
Index: RangeIndex(start=0, stop=5, step=1)


`.shape` returns `(rows, columns)` — 5 employees, 5 columns. `.columns` gives the column names. `.index` shows the row labels — a `RangeIndex` from 0 to 4, assigned automatically.

---

## 2.2 Basic Inspection Methods

Every new dataset gets the same treatment: `.head()`, `.tail()`, `.info()`, `.describe()`, `.dtypes`. Run all five before doing anything else.

In [4]:
# .head() — first N rows (default 5)
print(employees.head(3))

      name   department   salary   hire_date    city
0    Alice        Sales  50000.0  2021-03-15  Denver
1      Bob  Engineering  75000.0  2019-07-01  Austin
2  Charlie        Sales  55000.0  2022-01-10  Denver


In [5]:
# .tail() — last N rows
print(employees.tail(2))

     name   department   salary   hire_date     city
3  Diana            HR  62000.0  2020-11-20  Chicago
4     Eve  Engineering      NaN  2023-06-05   Austin


`.tail(2)` reveals Eve's missing salary immediately. With large datasets you cannot see every row, so `.head()` and `.tail()` are your first window into the data.

In [6]:
# .info() — the single most important inspection method
employees.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        5 non-null      object 
 1   department  5 non-null      object 
 2   salary      4 non-null      float64
 3   hire_date   5 non-null      object 
 4   city        5 non-null      object 
dtypes: float64(1), object(4)
memory usage: 328.0+ bytes


**What to notice from `.info()`:**

1. **salary has 4 non-null values** out of 5 rows — one value is missing (Eve's).
2. **salary is float64**, not int64 — the `None` forced pandas to convert the entire column to float.
3. **hire_date is `str`** — dates stored as strings cannot be sorted chronologically. Section 2.4 shows how to fix this.

In [7]:
# .describe() — summary statistics for numeric columns
print(employees.describe())

            salary
count      4.00000
mean   60500.00000
std    10847.42673
min    50000.00000
25%    53750.00000
50%    58500.00000
75%    65250.00000
max    75000.00000


**Reading hints for `.describe()`:**
- **count < total rows** means there are null values (count=4 but 5 rows = Eve's missing salary)
- **min much lower than 25%** can signal outliers on the low end
- **std much larger than mean** means high variance

In [8]:
# .dtypes — quick glance at column types
print(employees.dtypes)

name           object
department     object
salary        float64
hire_date      object
city           object
dtype: object


**Key insight:** In pandas, string columns typically appear as `object` (default) or `string` (StringDtype in modern pandas). When a column that should be numeric or datetime appears as `object` or `string`, it indicates a data type issue that should be fixed using `.astype()` or `pd.to_datetime()`.

---

## 2.3 Thinking Before Coding

Before writing any analysis code, answer three questions:

1. **What is each row?** — In the Acme data, each row is one employee.
2. **What are the columns?** — name (string), department (categorical), salary (numeric, USD, annual), hire_date (date as string), city (categorical).
3. **What is missing?** — Which columns have nulls? How many? Is the missingness random or systematic?

In [9]:
# Practice the three questions on the Acme employees DataFrame
print(f"Rows: {employees.shape[0]}")
print(f"Columns: {employees.shape[1]}")
print(f"Missing values per column:")
print(employees.isnull().sum())

Rows: 5
Columns: 5
Missing values per column:
name          0
department    0
salary        1
hire_date     0
city          0
dtype: int64


Your manager asks: *"How many employees do we have, and is the data complete?"*

You can now answer: "Five employees, but one salary is missing — Eve in Engineering."

**Key insight:** The three-question habit prevents you from writing code that answers the wrong question.

---

## 2.4 Common Data Type Issues

Two type problems appear constantly: numbers stored as strings and dates stored as strings.

In [10]:
# Numbers stored as strings — the $ and commas prevent numeric operations
orders = pd.DataFrame({
    'item': ['Widget', 'Gadget', 'Sprocket'],
    'price': ['$1,200', '$850', '$2,100']
})

print('Before fix:')
print(orders.dtypes)
print()

Before fix:
item     object
price    object
dtype: object



In [ ]:
# Fix: strip formatting, then convert
orders['price'] = orders['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False)

## can also be broken into two steps for clarity:
# orders['price'] = orders['price'].str.replace('$', '', regex=False)
# orders['price'] = orders['price'].str.replace(',', '', regex=False)

orders['price'] = orders['price'].astype(float)

print('After fix:')
print(orders.dtypes)
print()
print(orders)

After fix:
item      object
price    float64
dtype: object

       item   price
0    Widget  1200.0
1    Gadget   850.0
2  Sprocket  2100.0


In [12]:
# Dates stored as strings — fix with pd.to_datetime()
print('Before:', employees['hire_date'].dtype)

employees['hire_date'] = pd.to_datetime(employees['hire_date'])

print('After:', employees['hire_date'].dtype)
print()
print(employees.dtypes)

Before: object
After: datetime64[ns]

name                  object
department            object
salary               float64
hire_date     datetime64[ns]
city                  object
dtype: object


`hire_date` is now `datetime64[us]` — pandas understands it as a date. Chapter 15 covers time series operations in depth.

**Common mistake:** Assuming a column is the right type without checking. Zip codes like `['02101', '90210']` look numeric but should stay as `str` — converting to integer drops the leading zero.

---

## 2.5 Exporting Results

In [13]:
# Save to CSV — index=False prevents writing row numbers as an extra column
employees.to_csv('acme_employees.csv', index=False)
print('Saved acme_employees.csv')

Saved acme_employees.csv


In [14]:
# Verify the export by reading it back
reloaded = pd.read_csv('acme_employees.csv')
print('Shape:', reloaded.shape)
print()
print(reloaded.head())

Shape: (5, 5)

      name   department   salary   hire_date     city
0    Alice        Sales  50000.0  2021-03-15   Denver
1      Bob  Engineering  75000.0  2019-07-01   Austin
2  Charlie        Sales  55000.0  2022-01-10   Denver
3   Diana            HR  62000.0  2020-11-20  Chicago
4      Eve  Engineering      NaN  2023-06-05   Austin


**Note:** pandas supports many export formats — Excel, Parquet, JSON, SQL, and more. Chapter 4 covers them all. For now, `.to_csv()` with `index=False` is all you need.

---

## 2.6 When Things Go Wrong

Errors in pandas are normal. The skill is reading the error message, not avoiding errors entirely.

In [15]:
# Break this: mismatched list lengths
try:
    broken = pd.DataFrame({
        'name': ['Alice', 'Bob', 'Charlie'],
        'salary': [50000, 75000]
    })
except ValueError as e:
    print(f'ValueError: {e}')
    print()
    print('The last line tells you what went wrong — name has 3 values, salary has 2.')

ValueError: All arrays must be of the same length

The last line tells you what went wrong — name has 3 values, salary has 2.


In [16]:
# Break this: accessing a nonexistent column
small = pd.DataFrame({'name': ['Alice', 'Bob'], 'salary': [50000, 75000]})

try:
    print(small['department'])
except KeyError as e:
    print(f'KeyError: {e}')
    print()
    print('Check spelling, capitalization, and extra spaces.')

KeyError: 'department'

Check spelling, capitalization, and extra spaces.


### The IIVF Debugging Pattern

When something goes wrong, follow these four steps:

1. **Inspect** — look at `.head()`, `.dtypes`, `.shape`
2. **Isolate** — which line caused the error?
3. **Verify** — test a fix on a small slice
4. **Fix** — apply the fix and re-run

In [17]:
# IIVF in action: you expected a numeric operation but it fails
data = pd.DataFrame({'value': ['10', '20', '30']})

# STEP 1 — Inspect
print('Dtypes:', data.dtypes.to_dict())  # value is str, not int64

# STEP 2 — Isolate: the problem is the dtype, not the operation

# STEP 3 — Verify on a small piece
print('Can convert?', int('10'))  # works — the strings are valid integers

# STEP 4 — Fix
data['value'] = data['value'].astype(int)
print('Sum after fix:', data['value'].sum())  # 60

Dtypes: {'value': dtype('O')}
Can convert? 10
Sum after fix: 60


---

## 2.7 Common Beginner Mistakes

In [18]:
# Confusing = (assignment) and == (comparison)
df = pd.DataFrame({'status': ['active', 'inactive', 'active']})

# WRONG — this would reassign the column to the string 'active':
# df['status'] = 'active'

# RIGHT — this creates a boolean mask
mask = df['status'] == 'active'
print(mask)

0     True
1    False
2     True
Name: status, dtype: bool


In [19]:
# Modifying a copy instead of the original
df = pd.DataFrame({
    'name': ['Alice', 'Bob', None],
    'salary': [50000, 75000, 60000]
})

# WRONG — result is thrown away
df.dropna()
print('After df.dropna() without assignment:', df.shape)  # still (3, 2)

# RIGHT — assign the result
df_clean = df.dropna()
print('After df_clean = df.dropna():', df_clean.shape)    # (2, 2)

After df.dropna() without assignment: (3, 2)
After df_clean = df.dropna(): (2, 2)


**Key insight:** When a pandas method does not seem to work, check whether you assigned the result back. This is the single most common beginner mistake.

---

## 2.9 Key Takeaways

- **`.info()` is your X-ray** — run it on every new dataset before doing anything else
- **Run .shape, .dtypes, .head(), .info() on every new dataset** — catches missing data and type issues early
- **Three questions before any analysis:** What is each row? What are the columns? What is missing?
- **Numeric columns with `None` become float64** — expected pandas behavior, not a bug
- **Check dtypes early** — in pandas, string columns appear as `object` dtype, which can mask mixed-type issues.
- **Always export with `index=False`** — prevents an unwanted integer column in the CSV
- **IIVF: Inspect, Isolate, Verify, Fix** — debugging pattern for most pandas errors
- **Assign the result** — `df = df.dropna()`, not just `df.dropna()`

---

## Exercises

Complete the exercises below. Solutions are at the bottom of this notebook.

### Exercise 1: Create Your First DataFrame

Build the Acme Corp employees DataFrame from section 2.1. Then answer:
- How many rows and columns does it have?
- What are the column data types?
- Which column has a missing value?

Use `.shape`, `.dtypes`, and `.info()` to find the answers.

In [20]:
# Exercise 1 — Your code here
# TODO: Create the Acme Corp employees DataFrame
# TODO: Print .shape
# TODO: Print .dtypes
# TODO: Run .info()

### Exercise 2: Inspect a DataFrame with Basic Methods

Given the products DataFrame below, run `.head()`, `.tail(2)`, `.info()`, `.describe()`, and `.shape`. For each method, write one sentence explaining what you learned about the data.

In [21]:
# Exercise 2 — Your code here
products = pd.DataFrame({
    'product': ['Laptop', 'Mouse', 'Monitor', 'Keyboard', 'Headset'],
    'price': [1200, 25, 350, 75, 150],
    'stock': [15, 150, 42, 200, None],
    'category': ['Electronics', 'Accessories', 'Electronics', 'Accessories', 'Accessories']
})

# TODO: Run .head()
# TODO: Run .tail(2)
# TODO: Run .info()
# TODO: Run .describe()
# TODO: Print .shape
# TODO: Write one sentence per method about what you learned

### Exercise 3: Fix Common Data Type Issues

This DataFrame has two type problems — find and fix them:
1. Use `.dtypes` to identify which columns have the wrong type
2. Convert `salary` to a numeric type using `.astype()`
3. Convert `start_date` to a datetime type using `pd.to_datetime()`
4. Verify your fixes with `.dtypes`

In [22]:
# Exercise 3 — Your code here
messy = pd.DataFrame({
    'employee': ['Alice', 'Bob', 'Charlie'],
    'salary': ['50000', '75000', '55000'],
    'start_date': ['2021-03-15', '2019-07-01', '2022-01-10']
})

# TODO: Print .dtypes to see the problems
# TODO: Convert salary to float or int
# TODO: Convert start_date to datetime
# TODO: Print .dtypes to verify the fixes

### Exercise 4: Build a Simple Pandas Workflow

Create your own 5-column DataFrame with at least 6 rows. Include at least one numeric column, one date column, and one `None` value. Then:
1. Inspect with `.shape`, `.info()`, `.describe()`, `.dtypes`
2. Fix any data type issues
3. Export to CSV with `index=False`
4. Read the CSV back and verify with `.shape` and `.head()`

Answer the three questions from section 2.3: What is each row? What are the columns? What is missing?

In [23]:
# Exercise 4 — Your code here
# TODO: Create a 5-column, 6+ row DataFrame with a None and a date column
# TODO: Inspect with .shape, .info(), .describe(), .dtypes
# TODO: Fix any dtype issues
# TODO: Export to CSV with index=False
# TODO: Read back and verify
# TODO: Answer the three questions in comments or print statements

---

## Solutions

### Solution 1: Create Your First DataFrame

In [24]:
acme = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana ', 'Eve'],
    'department': ['Sales', 'Engineering', 'Sales', 'HR', 'Engineering'],
    'salary': [50000, 75000, 55000, 62000, None],
    'hire_date': ['2021-03-15', '2019-07-01', '2022-01-10', '2020-11-20', '2023-06-05'],
    'city': ['Denver', 'Austin', 'Denver', 'Chicago', 'Austin']
})

print('Shape:', acme.shape)    # (5, 5) — 5 rows, 5 columns
print()
print('Dtypes:')
print(acme.dtypes)             # salary is float64 (None forced conversion), hire_date is str
print()
acme.info()                    # salary has 4 non-null — one missing value (Eve's)

Shape: (5, 5)

Dtypes:
name           object
department     object
salary        float64
hire_date      object
city           object
dtype: object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        5 non-null      object 
 1   department  5 non-null      object 
 2   salary      4 non-null      float64
 3   hire_date   5 non-null      object 
 4   city        5 non-null      object 
dtypes: float64(1), object(4)
memory usage: 328.0+ bytes


**Answers:** 5 rows and 5 columns. salary is float64 (not int, because of the None), hire_date is str (not datetime). salary has a missing value (Eve).

### Solution 2: Inspect a DataFrame with Basic Methods

In [25]:
products = pd.DataFrame({
    'product': ['Laptop', 'Mouse', 'Monitor', 'Keyboard', 'Headset'],
    'price': [1200, 25, 350, 75, 150],
    'stock': [15, 150, 42, 200, None],
    'category': ['Electronics', 'Accessories', 'Electronics', 'Accessories', 'Accessories']
})

print('=== .head() ===')
print(products.head())
# All 5 rows visible — small dataset

print()
print('=== .tail(2) ===')
print(products.tail(2))
# Last two rows: Keyboard and Headset

print()
print('=== .info() ===')
products.info()
# stock has 4 non-null — Headset stock is missing. stock is float64 due to None.

print()
print('=== .describe() ===')
print(products.describe())
# count=4 for stock confirms one missing. price range: 25 to 1200.

print()
print('=== .shape ===')
print(products.shape)
# (5, 4) — 5 products, 4 attributes

=== .head() ===
    product  price  stock     category
0    Laptop   1200   15.0  Electronics
1     Mouse     25  150.0  Accessories
2   Monitor    350   42.0  Electronics
3  Keyboard     75  200.0  Accessories
4   Headset    150    NaN  Accessories

=== .tail(2) ===
    product  price  stock     category
3  Keyboard     75  200.0  Accessories
4   Headset    150    NaN  Accessories

=== .info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   product   5 non-null      object 
 1   price     5 non-null      int64  
 2   stock     4 non-null      float64
 3   category  5 non-null      object 
dtypes: float64(1), int64(1), object(2)
memory usage: 288.0+ bytes

=== .describe() ===
             price       stock
count     5.000000    4.000000
mean    360.000000  101.750000
std     485.605292   87.705473
min      25.000000   15.000000
25%      75.000000   

### Solution 3: Fix Common Data Type Issues

In [26]:
messy = pd.DataFrame({
    'employee': ['Alice', 'Bob', 'Charlie'],
    'salary': ['50000', '75000', '55000'],
    'start_date': ['2021-03-15', '2019-07-01', '2022-01-10']
})

print('Before fix:')
print(messy.dtypes)
# salary is str (should be numeric), start_date is str (should be datetime)

print()

# Fix salary
messy['salary'] = messy['salary'].astype(int)

# Fix start_date
messy['start_date'] = pd.to_datetime(messy['start_date'])

print('After fix:')
print(messy.dtypes)
# salary is int64, start_date is datetime64[us]

Before fix:
employee      object
salary        object
start_date    object
dtype: object

After fix:
employee              object
salary                 int64
start_date    datetime64[ns]
dtype: object


### Solution 4: Build a Simple Pandas Workflow

In [27]:
# 1. Create a custom DataFrame
students = pd.DataFrame({
    'name': ['Fatima', 'James', 'Yuki', 'Carlos', 'Priya', 'Noah'],
    'major': ['CS', 'Math', 'CS', 'Physics', 'Math', 'CS'],
    'gpa': [3.8, 3.5, None, 3.2, 3.9, 3.6],
    'grad_date': ['2024-05-15', '2024-05-15', '2025-05-15', '2024-12-15', '2025-05-15', '2024-05-15'],
    'age': [22, 24, 23, 25, 22, 21]
})

# 2. Inspect
print('Shape:', students.shape)
print()
students.info()
print()
print(students.describe())
print()
print(students.dtypes)

# Three questions:
# - Each row is one student
# - Columns: name, major, gpa (0-4.0), graduation date, age
# - gpa has 1 missing value (Yuki)

Shape: (6, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       6 non-null      object 
 1   major      6 non-null      object 
 2   gpa        5 non-null      float64
 3   grad_date  6 non-null      object 
 4   age        6 non-null      int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 368.0+ bytes

            gpa        age
count  5.000000   6.000000
mean   3.600000  22.833333
std    0.273861   1.471960
min    3.200000  21.000000
25%    3.500000  22.000000
50%    3.600000  22.500000
75%    3.800000  23.750000
max    3.900000  25.000000

name          object
major         object
gpa          float64
grad_date     object
age            int64
dtype: object


In [28]:
# 3. Fix dtype issues
students['grad_date'] = pd.to_datetime(students['grad_date'])
print('Fixed dtypes:')
print(students.dtypes)

# 4. Export and verify
students.to_csv('students.csv', index=False)
reloaded = pd.read_csv('students.csv')
print()
print('Reloaded shape:', reloaded.shape)
print()
print(reloaded.head())

Fixed dtypes:
name                 object
major                object
gpa                 float64
grad_date    datetime64[ns]
age                   int64
dtype: object

Reloaded shape: (6, 5)

     name    major  gpa   grad_date  age
0  Fatima       CS  3.8  2024-05-15   22
1   James     Math  3.5  2024-05-15   24
2    Yuki       CS  NaN  2025-05-15   23
3  Carlos  Physics  3.2  2024-12-15   25
4   Priya     Math  3.9  2025-05-15   22


**Answers to the three questions:**
- **What is each row?** One student.
- **What are the columns?** Name (object), major (categorical), GPA (numeric, 0-4.0 scale), graduation date (date), age (integer).
- **What is missing?** Yuki's GPA is missing (1 null in gpa column).